# 05_prepare_kqapro_selected_for_translation_pipeline

Этот ноутбук **не переводит** запросы сам.  
Он берёт вручную отобранные строки из `kqapro_manual_review_sheet.csv`, подтягивает полные записи из `kqapro_reconstruction_shortlist.jsonl`, размечает служебные поля и экспортирует **упакованный selected-layer** для дальнейшей загрузки в переводящий / рефразящий пайплайн.

## Что делает
1. Читает ручной CSV после отбора.
2. Определяет, какие строки считать выбранными.
3. Подтягивает полные записи из shortlist JSONL по `benchmark_id`.
4. Оценивает `answer_mode` (`single` / `count` / `boolean` / `list_like`).
5. Собирает единый экспортный датасет для инфры.
6. Пишет `jsonl/csv` + summary.

## Что не делает
- не переводит на русский;
- не делает rephrase;
- не делает dedup;
- не изменяет смысл вопросов.

## Ожидаемый результат
На выходе появятся файлы:
- `artifacts_kqapro_stage5/jsonl/kqapro_selected_for_translation_pipeline.jsonl`
- `artifacts_kqapro_stage5/csv/kqapro_selected_for_translation_pipeline.csv`
- `artifacts_kqapro_stage5/csv/kqapro_translation_minimal.csv`
- `artifacts_kqapro_stage5/reports/kqapro_stage5_summary.json`


In [1]:
# =========================
# УСТАНОВКА (если нужно)
# =========================
# %pip install -q pandas

In [2]:
# =========================
# ИМПОРТЫ И PATHS
# =========================
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

PROJECT_ROOT = Path(".").resolve()

STAGE3_DIR = PROJECT_ROOT / "artifacts_kqapro_stage3"
REVIEW_CSV_PATH = STAGE3_DIR / "csv" / "kqapro_manual_review_sheet.csv"
SHORTLIST_JSONL_PATH = STAGE3_DIR / "jsonl" / "kqapro_reconstruction_shortlist.jsonl"

OUTPUT_DIR = PROJECT_ROOT / "artifacts_kqapro_stage5"
JSONL_DIR = OUTPUT_DIR / "jsonl"
CSV_DIR = OUTPUT_DIR / "csv"
REPORTS_DIR = OUTPUT_DIR / "reports"

for p in [JSONL_DIR, CSV_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("REVIEW_CSV_PATH      =", REVIEW_CSV_PATH)
print("SHORTLIST_JSONL_PATH =", SHORTLIST_JSONL_PATH)
print("OUTPUT_DIR           =", OUTPUT_DIR)


REVIEW_CSV_PATH      = /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage3/csv/kqapro_manual_review_sheet.csv
SHORTLIST_JSONL_PATH = /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage3/jsonl/kqapro_reconstruction_shortlist.jsonl
OUTPUT_DIR           = /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage5


In [3]:
# =========================
# БАЗОВЫЕ УТИЛИТЫ
# =========================
def safe_str(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    return str(x).strip()

def safe_lower(x: Any) -> str:
    return safe_str(x).strip().lower()

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def normalize_colnames(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [safe_str(c) for c in out.columns]
    return out

def extract_source_split(benchmark_id: str) -> str:
    bid = safe_lower(benchmark_id)
    if "_train_" in bid or bid.startswith("kqapro_train_"):
        return "train"
    if "_val_" in bid or bid.startswith("kqapro_val_"):
        return "val"
    if "_test_" in bid or bid.startswith("kqapro_test_"):
        return "test"
    return "unknown"

def infer_answer_mode(question_en: str, answer_text: str, reasoning_family: str) -> str:
    q = safe_lower(question_en)
    a = safe_lower(answer_text)
    fam = safe_lower(reasoning_family)

    if fam == "count" or q.startswith("how many") or q.startswith("how much") or q.startswith("what number"):
        return "count"

    if q.startswith(("is ", "was ", "were ", "does ", "did ", "do ", "has ", "have ", "had ", "can ", "could ", "should ", "will ", "would ", "are ")):
        return "boolean"

    if a in {"yes", "no", "true", "false"}:
        return "boolean"

    # Очень грубая эвристика на список ответов
    if any(sep in answer_text for sep in [" | ", " ; ", "; ", "\t"]) or answer_text.count(",") >= 2:
        return "list_like"

    return "single"

def make_translation_instruction(answer_mode: str) -> str:
    common = (
        "Translate the question from English to Russian while preserving entities, "
        "numbers, dates, ranges, comparisons, and logical constraints exactly. "
        "Do not simplify the logic and do not add new conditions."
    )
    if answer_mode == "count":
        return common + " Keep the question in a counting form."
    if answer_mode == "boolean":
        return common + " Keep the question in a yes/no form."
    if answer_mode == "list_like":
        return common + " Preserve the list-oriented nature of the question."
    return common + " Preserve the original answer mode."


In [4]:
# =========================
# ЗАГРУЗКА REVIEW CSV
# =========================
if not REVIEW_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Не найден review CSV: {REVIEW_CSV_PATH}\n"
        "Укажи правильный путь или положи файл в artifacts_kqapro_stage3/csv/"
    )

review_df = pd.read_csv(REVIEW_CSV_PATH)
review_df = normalize_colnames(review_df)

print("review_df.shape =", review_df.shape)
print("review_df.columns =", review_df.columns.tolist())
review_df.head(3)


review_df.shape = (74, 14)
review_df.columns = ['benchmark_id', 'primary_reasoning_family', 'recommended_action', 'reconstructability_score_v2', 'program_signature', 'program_length', 'hop_count_estimate', 'question_en_original', 'answer_text', 'manual_decision', 'manual_bucket', 'russian_reconstruction_idea', 'target_domain_ru', 'notes']


,benchmark_id,primary_reasoning_family,recommended_action,reconstructability_score_v2,program_signature,program_length,hop_count_estimate,question_en_original,answer_text,manual_decision,manual_bucket,russian_reconstruction_idea,target_domain_ru,notes
0,kqapro_train_013987,multi_hop,keep_for_reconstruction,26,Find -> Relate -> QFilterYear -> FilterConcept...,12,3,"Which big city was, in 1958, a twinned adminis...",Marseille,NaN,NaN,NaN,NaN,NaN
1,kqapro_train_035873,multi_hop,keep_for_reconstruction,26,Find -> Relate -> Find -> And -> Relate -> Fil...,12,3,"Who is a cast member of ""Taken"" (which also ha...",Michael Jeter,NaN,NaN,NaN,NaN,NaN
2,kqapro_train_075279,multi_hop,keep_for_reconstruction,26,Find -> Relate -> Find -> And -> Relate -> Fil...,10,3,"What person is a member, starting in 2012, of ...",Thiago Silva,NaN,NaN,NaN,NaN,NaN


In [ ]:
# =========================
# ОПРЕДЕЛЕНИЕ ВЫБРАННЫХ СТРОК
# =========================

KEEP_MARKERS = {
    "keep", "yes", "selected", "select", "1", "true",
    "keep_for_reconstruction", "core_list", "aux_single", "pattern_only"
}

def is_keep_marker(x: Any) -> bool:
    return safe_lower(x) in KEEP_MARKERS

review_df["_keep_from_decision"] = False
review_df["_keep_from_bucket"] = False
review_df["_keep_from_flag"] = False

if "manual_decision" in review_df.columns:
    review_df["_keep_from_decision"] = review_df["manual_decision"].apply(is_keep_marker)

if "manual_bucket" in review_df.columns:
    review_df["_keep_from_bucket"] = review_df["manual_bucket"].apply(is_keep_marker)

for flag_col in ["selected", "keep", "include", "is_selected"]:
    if flag_col in review_df.columns:
        review_df["_keep_from_flag"] = review_df[flag_col].apply(is_keep_marker)
        break

selected_mask = (
    review_df["_keep_from_decision"] |
    review_df["_keep_from_bucket"] |
    review_df["_keep_from_flag"]
)

num_marked = int(selected_mask.sum())

if num_marked == 0:
    print("[select] Явных keep-маркеров не найдено.")
    print("[select] Будут использованы ВСЕ строки из review CSV как уже вручную отобранный список.")
    selected_df = review_df.copy()
else:
    print(f"[select] Найдено {num_marked} строк с keep-маркерами.")
    selected_df = review_df[selected_mask].copy()

selected_df = selected_df.reset_index(drop=True)

if "benchmark_id" not in selected_df.columns:
    raise KeyError("В review CSV нет колонки benchmark_id")

print("selected_df.shape =", selected_df.shape)
selected_df.head(5)


[select] Явных keep-маркеров не найдено.
[select] Будут использованы ВСЕ строки из review CSV как уже вручную отобранный список.
selected_df.shape = (74, 17)


,benchmark_id,primary_reasoning_family,recommended_action,reconstructability_score_v2,program_signature,program_length,hop_count_estimate,question_en_original,answer_text,manual_decision,manual_bucket,russian_reconstruction_idea,target_domain_ru,notes,_keep_from_decision,_keep_from_bucket,_keep_from_flag
0,kqapro_train_013987,multi_hop,keep_for_reconstruction,26,Find -> Relate -> QFilterYear -> FilterConcept...,12,3,"Which big city was, in 1958, a twinned adminis...",Marseille,NaN,NaN,NaN,NaN,NaN,False,False,False
1,kqapro_train_035873,multi_hop,keep_for_reconstruction,26,Find -> Relate -> Find -> And -> Relate -> Fil...,12,3,"Who is a cast member of ""Taken"" (which also ha...",Michael Jeter,NaN,NaN,NaN,NaN,NaN,False,False,False
2,kqapro_train_075279,multi_hop,keep_for_reconstruction,26,Find -> Relate -> Find -> And -> Relate -> Fil...,10,3,"What person is a member, starting in 2012, of ...",Thiago Silva,NaN,NaN,NaN,NaN,NaN,False,False,False
3,kqapro_train_075408,multi_hop,keep_for_reconstruction,25,FindAll -> FilterNum -> FilterConcept -> Find ...,12,2,Which Japanese prefecture consists of more tha...,Ōsaka Prefecture,NaN,NaN,NaN,NaN,NaN,False,False,False
4,kqapro_train_076295,multi_hop,keep_for_reconstruction,25,Find -> Relate -> Find -> And -> Relate -> Fil...,12,2,What New York county in New York (the one that...,Greene County,NaN,NaN,NaN,NaN,NaN,False,False,False


In [6]:
# =========================
# УБИРАЕМ ПОВТОРЫ ПО benchmark_id
# =========================
before = len(selected_df)
selected_df["benchmark_id"] = selected_df["benchmark_id"].apply(safe_str)
selected_df = selected_df[selected_df["benchmark_id"] != ""].copy()
selected_df = selected_df.drop_duplicates(subset=["benchmark_id"], keep="first").reset_index(drop=True)
after = len(selected_df)

print(f"before={before}, after_dedup_by_id={after}, removed={before - after}")
selected_df[["benchmark_id"]].head(10)


before=74, after_dedup_by_id=72, removed=2


,benchmark_id
0,kqapro_train_013987
1,kqapro_train_035873
2,kqapro_train_075279
3,kqapro_train_075408
4,kqapro_train_076295
5,kqapro_train_035011
6,kqapro_train_085057
7,kqapro_train_090128
8,kqapro_val_007523
9,kqapro_train_080370


In [7]:
# =========================
# ЗАГРУЗКА FULL SHORTLIST JSONL
# =========================
if not SHORTLIST_JSONL_PATH.exists():
    raise FileNotFoundError(
        f"Не найден shortlist JSONL: {SHORTLIST_JSONL_PATH}\n"
        "Укажи правильный путь или сначала прогоняй stage3 notebook."
    )

shortlist_rows = read_jsonl(SHORTLIST_JSONL_PATH)
shortlist_df = pd.DataFrame(shortlist_rows)
shortlist_df = normalize_colnames(shortlist_df)

if "benchmark_id" not in shortlist_df.columns:
    raise KeyError("В shortlist JSONL нет колонки benchmark_id")

shortlist_df["benchmark_id"] = shortlist_df["benchmark_id"].apply(safe_str)
shortlist_df = shortlist_df.drop_duplicates(subset=["benchmark_id"], keep="first").reset_index(drop=True)

print("shortlist_df.shape =", shortlist_df.shape)
shortlist_df.head(3)


shortlist_df.shape = (180, 34)


,benchmark_id,source_dataset,source_split,import_mode,role,question_ru,question_en_original,domain,reasoning_tags,primary_reasoning_family,...,reconstruction_prompt,notes,drop_reasons,num_drop_reasons,hard_keep,reconstructability_score_v2,recommended_action,question_len,num_parentheses,list_style_friendly
0,kqapro_train_013987,kqa_pro,train,template_transfer,template_transfer_candidate,None,"Which big city was, in 1958, a twinned adminis...",,"[filtering, high_compositional_depth, logical_...",multi_hop,...,Сгенерируй естественный запрос на русском язык...,,[],0,True,26,keep_for_reconstruction,212,2,True
1,kqapro_train_088566,kqa_pro,train,template_transfer,template_transfer_candidate,None,What musical received Grammy Award for Best Mu...,,"[filtering, high_compositional_depth, logical_...",multi_hop,...,Сгенерируй естественный запрос на русском язык...,,[],0,True,26,keep_for_reconstruction,149,2,True
2,kqapro_train_035873,kqa_pro,train,template_transfer,template_transfer_candidate,None,Who is a cast member of Taken (which also has ...,,"[filtering, high_compositional_depth, logical_...",multi_hop,...,Сгенерируй естественный запрос на русском язык...,,[],0,True,26,keep_for_reconstruction,149,2,True


In [8]:
# =========================
# JOIN: SELECTED CSV + FULL JSONL
# =========================
selected_joined = selected_df.merge(
    shortlist_df,
    on="benchmark_id",
    how="left",
    suffixes=("_review", "_full"),
)

print("selected_joined.shape =", selected_joined.shape)

missing_full = selected_joined["question_en_original_full"].isna().sum() if "question_en_original_full" in selected_joined.columns else len(selected_joined)
print("missing_full_records =", int(missing_full))
selected_joined.head(3)


selected_joined.shape = (72, 50)
missing_full_records = 0


,benchmark_id,primary_reasoning_family_review,recommended_action_review,reconstructability_score_v2_review,program_signature_review,program_length_review,hop_count_estimate_review,question_en_original_review,answer_text_review,manual_decision,...,reconstruction_prompt,notes_full,drop_reasons,num_drop_reasons,hard_keep,reconstructability_score_v2_full,recommended_action_full,question_len,num_parentheses,list_style_friendly
0,kqapro_train_013987,multi_hop,keep_for_reconstruction,26,Find -> Relate -> QFilterYear -> FilterConcept...,12,3,"Which big city was, in 1958, a twinned adminis...",Marseille,NaN,...,Сгенерируй естественный запрос на русском язык...,,[],0,True,26,keep_for_reconstruction,212,2,True
1,kqapro_train_035873,multi_hop,keep_for_reconstruction,26,Find -> Relate -> Find -> And -> Relate -> Fil...,12,3,"Who is a cast member of ""Taken"" (which also ha...",Michael Jeter,NaN,...,Сгенерируй естественный запрос на русском язык...,,[],0,True,26,keep_for_reconstruction,149,2,True
2,kqapro_train_075279,multi_hop,keep_for_reconstruction,26,Find -> Relate -> Find -> And -> Relate -> Fil...,10,3,"What person is a member, starting in 2012, of ...",Thiago Silva,NaN,...,Сгенерируй естественный запрос на русском язык...,,[],0,True,26,keep_for_reconstruction,167,2,True


In [9]:
# =========================
# НОРМАЛИЗАЦИЯ ПОЛЕЙ
# =========================
def pick_col(row: pd.Series, base: str) -> Any:
    review_name = f"{base}_review"
    full_name = f"{base}_full"

    if full_name in row.index:
        val = row[full_name]
        if safe_str(val) != "":
            return val
    if review_name in row.index:
        val = row[review_name]
        if safe_str(val) != "":
            return val
    if base in row.index:
        return row[base]
    return None

prepared_rows = []

for _, row in selected_joined.iterrows():
    benchmark_id = safe_str(row.get("benchmark_id"))

    question_en_original = safe_str(pick_col(row, "question_en_original"))
    answer_text = safe_str(pick_col(row, "answer_text"))
    reasoning_family = safe_str(pick_col(row, "primary_reasoning_family"))
    program_signature = safe_str(pick_col(row, "program_signature"))
    domain = safe_str(pick_col(row, "domain"))
    target_domain_ru = safe_str(row.get("target_domain_ru"))
    manual_decision = safe_str(row.get("manual_decision"))
    manual_bucket = safe_str(row.get("manual_bucket"))
    russian_reconstruction_idea = safe_str(row.get("russian_reconstruction_idea"))
    notes = safe_str(row.get("notes"))

    program_length = pd.to_numeric(pick_col(row, "program_length"), errors="coerce")
    hop_count_estimate = pd.to_numeric(pick_col(row, "hop_count_estimate"), errors="coerce")
    reconstructability_score_v2 = pd.to_numeric(pick_col(row, "reconstructability_score_v2"), errors="coerce")

    answer_mode = infer_answer_mode(question_en_original, answer_text, reasoning_family)
    source_split = extract_source_split(benchmark_id)

    prepared_rows.append({
        "source_id": benchmark_id,
        "benchmark_id": benchmark_id,
        "source_dataset": "kqapro",
        "source_split": source_split,
        "question_en_original": question_en_original,
        "answer_text": answer_text,
        "answer_mode": answer_mode,
        "primary_reasoning_family": reasoning_family,
        "program_signature": program_signature,
        "program_length": None if pd.isna(program_length) else int(program_length),
        "hop_count_estimate": None if pd.isna(hop_count_estimate) else int(hop_count_estimate),
        "reconstructability_score_v2": None if pd.isna(reconstructability_score_v2) else int(reconstructability_score_v2),
        "domain": domain,
        "target_domain_ru": target_domain_ru,
        "manual_decision": manual_decision,
        "manual_bucket": manual_bucket,
        "russian_reconstruction_idea": russian_reconstruction_idea,
        "notes": notes,
        "needs_translation": True,
        "needs_rephrase": True,
        "translation_source_text": question_en_original,
        "translation_target_language": "ru",
        "translation_instruction": make_translation_instruction(answer_mode),
        "translation_stage": "selected_kqapro_to_ru",
        "rephrase_source_language": "ru",
        "rephrase_mode": "humanize_after_translation",
    })

prepared_df = pd.DataFrame(prepared_rows)
print("prepared_df.shape =", prepared_df.shape)
prepared_df.head(5)


prepared_df.shape = (72, 26)


,source_id,benchmark_id,source_dataset,source_split,question_en_original,answer_text,answer_mode,primary_reasoning_family,program_signature,program_length,...,russian_reconstruction_idea,notes,needs_translation,needs_rephrase,translation_source_text,translation_target_language,translation_instruction,translation_stage,rephrase_source_language,rephrase_mode
0,kqapro_train_013987,kqapro_train_013987,kqapro,train,"Which big city was, in 1958, a twinned adminis...",Marseille,single,multi_hop,Find -> Relate -> QFilterYear -> FilterConcept...,12,...,,,True,True,"Which big city was, in 1958, a twinned adminis...",ru,Translate the question from English to Russian...,selected_kqapro_to_ru,ru,humanize_after_translation
1,kqapro_train_035873,kqapro_train_035873,kqapro,train,Who is a cast member of Taken (which also has ...,Michael Jeter,single,multi_hop,Find -> Relate -> Find -> And -> Relate -> Fil...,12,...,,,True,True,Who is a cast member of Taken (which also has ...,ru,Translate the question from English to Russian...,selected_kqapro_to_ru,ru,humanize_after_translation
2,kqapro_train_075279,kqapro_train_075279,kqapro,train,"What person is a member, starting in 2012, of ...",Thiago Silva,single,multi_hop,Find -> Relate -> Find -> And -> Relate -> Fil...,10,...,,,True,True,"What person is a member, starting in 2012, of ...",ru,Translate the question from English to Russian...,selected_kqapro_to_ru,ru,humanize_after_translation
3,kqapro_train_075408,kqapro_train_075408,kqapro,train,Which Japanese prefecture consists of more tha...,Ōsaka Prefecture,single,multi_hop,FindAll -> FilterNum -> FilterConcept -> Find ...,12,...,,,True,True,Which Japanese prefecture consists of more tha...,ru,Translate the question from English to Russian...,selected_kqapro_to_ru,ru,humanize_after_translation
4,kqapro_train_076295,kqapro_train_076295,kqapro,train,What New York county in New York (the one that...,Greene County,single,multi_hop,Find -> Relate -> Find -> And -> Relate -> Fil...,12,...,,,True,True,What New York county in New York (the one that...,ru,Translate the question from English to Russian...,selected_kqapro_to_ru,ru,humanize_after_translation


In [10]:
# =========================
# БЫСТРАЯ ВАЛИДАЦИЯ
# =========================
assert prepared_df["source_id"].nunique() == len(prepared_df), "source_id должны быть уникальны"
assert prepared_df["question_en_original"].map(lambda x: safe_str(x) != "").all(), "Пустые question_en_original обнаружены"

mode_counts = prepared_df["answer_mode"].value_counts(dropna=False).reset_index()
mode_counts.columns = ["answer_mode", "count"]

family_counts = prepared_df["primary_reasoning_family"].value_counts(dropna=False).reset_index()
family_counts.columns = ["primary_reasoning_family", "count"]

print("Answer modes:")
display(mode_counts)

print("Reasoning families:")
display(family_counts)

print("Preview:")
display(prepared_df[[
    "source_id",
    "primary_reasoning_family",
    "answer_mode",
    "question_en_original",
    "answer_text",
    "target_domain_ru",
]].head(20))


Answer modes:


,answer_mode,count
0,single,72


Reasoning families:


,primary_reasoning_family,count
0,qualifier,21
1,temporal,17
2,logical_set,15
3,multi_hop,14
4,comparison_or_superlative,5


Preview:


,source_id,primary_reasoning_family,answer_mode,question_en_original,answer_text,target_domain_ru
0,kqapro_train_013987,multi_hop,single,"Which big city was, in 1958, a twinned adminis...",Marseille,
1,kqapro_train_035873,multi_hop,single,Who is a cast member of Taken (which also has ...,Michael Jeter,
2,kqapro_train_075279,multi_hop,single,"What person is a member, starting in 2012, of ...",Thiago Silva,
3,kqapro_train_075408,multi_hop,single,Which Japanese prefecture consists of more tha...,Ōsaka Prefecture,
4,kqapro_train_076295,multi_hop,single,What New York county in New York (the one that...,Greene County,
5,kqapro_train_035011,multi_hop,single,Which visual artwork has its narrative locatio...,Face/Off,
6,kqapro_train_085057,multi_hop,single,Which television series contains Six Feet Unde...,Everybody Loves Raymond,
7,kqapro_train_090128,multi_hop,single,What person is the winner of the Screen Actors...,Jack Huston,
8,kqapro_val_007523,multi_hop,single,Which person received the Cannes Film Festival...,Christoph Waltz,
9,kqapro_train_080370,multi_hop,single,What is the beginning date that Viacom (that i...,1986,


## Минимальный экспорт

Помимо полного датасета будет собран и минимальный экспорт, который удобно грузить в translation/rephrase pipeline как плоскую таблицу.


In [11]:
# =========================
# MINIMAL EXPORT
# =========================
minimal_cols = [
    "source_id",
    "source_dataset",
    "source_split",
    "question_en_original",
    "answer_mode",
    "primary_reasoning_family",
    "target_domain_ru",
    "needs_translation",
    "needs_rephrase",
    "translation_source_text",
    "translation_target_language",
    "translation_instruction",
    "translation_stage",
]

minimal_df = prepared_df[minimal_cols].copy()
print("minimal_df.shape =", minimal_df.shape)
minimal_df.head(5)


minimal_df.shape = (72, 13)


,source_id,source_dataset,source_split,question_en_original,answer_mode,primary_reasoning_family,target_domain_ru,needs_translation,needs_rephrase,translation_source_text,translation_target_language,translation_instruction,translation_stage
0,kqapro_train_013987,kqapro,train,"Which big city was, in 1958, a twinned adminis...",single,multi_hop,,True,True,"Which big city was, in 1958, a twinned adminis...",ru,Translate the question from English to Russian...,selected_kqapro_to_ru
1,kqapro_train_035873,kqapro,train,Who is a cast member of Taken (which also has ...,single,multi_hop,,True,True,Who is a cast member of Taken (which also has ...,ru,Translate the question from English to Russian...,selected_kqapro_to_ru
2,kqapro_train_075279,kqapro,train,"What person is a member, starting in 2012, of ...",single,multi_hop,,True,True,"What person is a member, starting in 2012, of ...",ru,Translate the question from English to Russian...,selected_kqapro_to_ru
3,kqapro_train_075408,kqapro,train,Which Japanese prefecture consists of more tha...,single,multi_hop,,True,True,Which Japanese prefecture consists of more tha...,ru,Translate the question from English to Russian...,selected_kqapro_to_ru
4,kqapro_train_076295,kqapro,train,What New York county in New York (the one that...,single,multi_hop,,True,True,What New York county in New York (the one that...,ru,Translate the question from English to Russian...,selected_kqapro_to_ru


In [12]:
# =========================
# ЭКСПОРТ
# =========================
full_jsonl_path = JSONL_DIR / "kqapro_selected_for_translation_pipeline.jsonl"
full_json_path = JSONL_DIR / "kqapro_selected_for_translation_pipeline.json"
full_csv_path = CSV_DIR / "kqapro_selected_for_translation_pipeline.csv"

minimal_jsonl_path = JSONL_DIR / "kqapro_translation_minimal.jsonl"
minimal_json_path = JSONL_DIR / "kqapro_translation_minimal.json"
minimal_csv_path = CSV_DIR / "kqapro_translation_minimal.csv"

write_jsonl(full_jsonl_path, prepared_df.to_dict("records"))
prepared_df.to_csv(full_csv_path, index=False)
with open(full_json_path, "w", encoding="utf-8") as f:
    json.dump(prepared_df.to_dict("records"), f, ensure_ascii=False, indent=2)

write_jsonl(minimal_jsonl_path, minimal_df.to_dict("records"))
minimal_df.to_csv(minimal_csv_path, index=False)
with open(minimal_json_path, "w", encoding="utf-8") as f:
    json.dump(minimal_df.to_dict("records"), f, ensure_ascii=False, indent=2)

summary = {
    "review_csv_path": str(REVIEW_CSV_PATH),
    "shortlist_jsonl_path": str(SHORTLIST_JSONL_PATH),
    "num_review_rows": int(len(review_df)),
    "num_selected_rows": int(len(selected_df)),
    "num_packaged_rows": int(len(prepared_df)),
    "num_missing_full_records_after_join": int(
        selected_joined["question_en_original_full"].isna().sum()
    ) if "question_en_original_full" in selected_joined.columns else None,
    "answer_mode_distribution": prepared_df["answer_mode"].value_counts().to_dict(),
    "reasoning_family_distribution": prepared_df["primary_reasoning_family"].value_counts().to_dict(),
    "exports": {
        "full_jsonl": str(full_jsonl_path),
        "full_json": str(full_json_path),
        "full_csv": str(full_csv_path),
        "minimal_jsonl": str(minimal_jsonl_path),
        "minimal_json": str(minimal_json_path),
        "minimal_csv": str(minimal_csv_path),
    },
}

summary_path = REPORTS_DIR / "kqapro_stage5_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("saved full JSONL    ->", full_jsonl_path)
print("saved full JSON     ->", full_json_path)
print("saved full CSV      ->", full_csv_path)
print("saved minimal JSONL ->", minimal_jsonl_path)
print("saved minimal JSON  ->", minimal_json_path)
print("saved minimal CSV   ->", minimal_csv_path)
print("saved summary       ->", summary_path)


saved full JSONL    -> /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage5/jsonl/kqapro_selected_for_translation_pipeline.jsonl
saved full JSON     -> /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage5/jsonl/kqapro_selected_for_translation_pipeline.json
saved full CSV      -> /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage5/csv/kqapro_selected_for_translation_pipeline.csv
saved minimal JSONL -> /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage5/jsonl/kqapro_translation_minimal.jsonl
saved minimal JSON  -> /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage5/jsonl/kqapro_translation_minimal.json
saved minimal CSV   -> /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage5/csv/kqapro_translation_minimal.csv
saved summary       -> /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage5/reports/kqapro_stage5_summary.json


In [13]:
# =========================
# CSV -> JSON КОНВЕРТЕР (опционально)
# =========================
def csv_to_json(csv_path: Path, json_path: Path) -> None:
    df = pd.read_csv(csv_path)
    records = df.to_dict("records")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"saved JSON -> {json_path}")

# примеры:
# csv_to_json(CSV_DIR / "kqapro_selected_for_translation_pipeline.csv",
#             JSONL_DIR / "kqapro_selected_for_translation_pipeline.from_csv.json")
# csv_to_json(CSV_DIR / "kqapro_translation_minimal.csv",
#             JSONL_DIR / "kqapro_translation_minimal.from_csv.json")


In [14]:
# =========================
# ФИНАЛЬНЫЙ PREVIEW
# =========================
display(prepared_df[[
    "source_id",
    "question_en_original",
    "answer_text",
    "answer_mode",
    "primary_reasoning_family",
    "program_signature",
    "target_domain_ru",
    "needs_translation",
    "needs_rephrase",
]].head(30))


,source_id,question_en_original,answer_text,answer_mode,primary_reasoning_family,program_signature,target_domain_ru,needs_translation,needs_rephrase
0,kqapro_train_013987,"Which big city was, in 1958, a twinned adminis...",Marseille,single,multi_hop,Find -> Relate -> QFilterYear -> FilterConcept...,,True,True
1,kqapro_train_035873,Who is a cast member of Taken (which also has ...,Michael Jeter,single,multi_hop,Find -> Relate -> Find -> And -> Relate -> Fil...,,True,True
2,kqapro_train_075279,"What person is a member, starting in 2012, of ...",Thiago Silva,single,multi_hop,Find -> Relate -> Find -> And -> Relate -> Fil...,,True,True
3,kqapro_train_075408,Which Japanese prefecture consists of more tha...,Ōsaka Prefecture,single,multi_hop,FindAll -> FilterNum -> FilterConcept -> Find ...,,True,True
4,kqapro_train_076295,What New York county in New York (the one that...,Greene County,single,multi_hop,Find -> Relate -> Find -> And -> Relate -> Fil...,,True,True
5,kqapro_train_035011,Which visual artwork has its narrative locatio...,Face/Off,single,multi_hop,Find -> Relate -> Find -> And -> Relate -> Fil...,,True,True
6,kqapro_train_085057,Which television series contains Six Feet Unde...,Everybody Loves Raymond,single,multi_hop,Find -> Relate -> Find -> And -> Relate -> QFi...,,True,True
7,kqapro_train_090128,What person is the winner of the Screen Actors...,Jack Huston,single,multi_hop,Find -> Relate -> QFilterYear -> FilterConcept...,,True,True
8,kqapro_val_007523,Which person received the Cannes Film Festival...,Christoph Waltz,single,multi_hop,Find -> Relate -> QFilterStr -> FilterConcept ...,,True,True
9,kqapro_train_080370,What is the beginning date that Viacom (that i...,1986,single,multi_hop,Find -> Relate -> Find -> And -> Find -> Relat...,,True,True
